# Predictive Maintenance AI - Exploratory Data Analysis (NASA C-MAPSS)
Requires `train_FD001.txt`, `test_FD001.txt`, `RUL_FD001.txt` in `data/`. Every number below is computed from those files.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
from src import config
from src.data_loader import load_subset, missing_report
from src.eda import run_eda, sensor_rul_correlation
from src.features import FeatureBuilder, split_engines
SUBSET = "FD001"
train, test = load_subset(SUBSET)
train.head()

## 1. Structure and missing values

In [ ]:
print(train.shape, test.shape)
print("Train missing:", missing_report(train))
print("Test missing:", missing_report(test))
train.describe().T[["mean", "std", "min", "max"]]

## 2. Engine-wise time series

In [ ]:
life = train.groupby("unit").cycle.max()
print(life.describe())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for u in life.index[:5]:
    e = train[train.unit == u]
    axes[0].plot(e.cycle, e.s11, lw=1, label=f"unit {u}")
    axes[1].plot(e.cycle, e.rul, lw=1)
axes[0].set(title="s11 over each engine's life", xlabel="cycle"); axes[0].legend()
axes[1].set(title="RUL label (capped piecewise-linear)", xlabel="cycle")
plt.show()

## 3. Sensor analysis

In [ ]:
std = train[config.SENSORS].std().sort_values()
print("Near-constant sensors:", std[std < 1e-4].index.tolist())
sensor_rul_correlation(train, std[std >= 1e-4].index.tolist())

## 4. Saved EDA figures

In [ ]:
fb = FeatureBuilder(SUBSET).fit(split_engines(train)[0])
summary = run_eda(train, fb.sensors, SUBSET)
for p in sorted(config.FIGURES_DIR.glob("0[1-6]_*.png")):
    display(Image(filename=str(p)))
summary

## 5. Engineered features (past-only rolling statistics)

In [ ]:
X = fb.transform(train[train.unit == 1].reset_index(drop=True))
print(len(fb.feature_names), "features")
X.head()

## 6. After `python train.py`: results

In [ ]:
for p in [config.REG_COMPARISON_PATH, config.CLF_COMPARISON_PATH]:
    if p.exists():
        display(pd.read_csv(p))
for name in ["07_rul_pred_vs_true.png", "08_rul_model_comparison.png", "09_failure_confusion_matrices.png",
             "10_failure_roc.png", "11_sensor_importance.png"]:
    p = config.FIGURES_DIR / name
    if p.exists():
        display(Image(filename=str(p)))